# 🇮🇳 VoiceGuard India - Ultimate AWS Master Pipeline

This is the final, fully-automated pipeline. It handles AWS SysOps (installing PyTorch to the EBS volume), downloads the HuggingFace datasets to the 100GB hard drive, and trains the model in the correct Python order.

In [ ]:
# ==========================================
# 1. SYS-OPS: INSTALL DECODERS TO 100GB DRIVE
# ==========================================
import os
print("Installing Audio Decoders to AWS 100GB Drive (Bypassing OS Limits)...")
os.system("mkdir -p ./aws_libs")
os.system("mkdir -p ./pip_cache")
os.system("TMPDIR=./pip_cache pip install --target ./aws_libs torchcodec 'datasets[audio]' --cache-dir ./pip_cache")
print("\n✅ Decoders Successfully Installed!")

In [ ]:
# ==========================================
# 2. INITIALIZATION (Force Python to use new decoders)
# ==========================================
import sys
sys.path.insert(0, "./aws_libs")

import librosa
import numpy as np
import pickle
import xgboost as xgb
from sklearn.metrics import classification_report
from tqdm import tqdm
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# ⚠️ ENTER YOUR HUGGINGFACE TOKEN HERE ⚠️
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

def safe_load(path, *args, **kwargs):
    try:
        return load_dataset(path, *args, token=HF_TOKEN, cache_dir="./hf_cache", **kwargs)
    except Exception as e:
        print(f"⚠️ Error loading {path}: {e}")
        return None

In [ ]:
# ==========================================
# 3. DOWNLOAD DATASETS TO AWS 100GB DRIVE
# ==========================================
print("\n🌍 Downloading HuggingFace Datasets to AWS Disk...")
auth_1 = safe_load("google/fleurs", "hi_in", split="train")
auth_2 = safe_load("SPRINGLab/IndicTTS-Hindi", split="train")
auth_3 = safe_load("ud-nlp/hindi-speech-recognition-dataset", split="train")
fake_2 = safe_load("abhirl/hindi-tts-dataset", split="train")
print("✅ Datasets Successfully Downloaded!")

In [ ]:
# ==========================================
# 4. EXTRACT MATHEMATICAL FEATURES
# ==========================================
def extract_features_from_array(audio_array, sr):
    try:
        mfccs = np.mean(librosa.feature.mfcc(y=audio_array, sr=sr, n_mfcc=20).T, axis=0)
        centroid = np.mean(librosa.feature.spectral_centroid(y=audio_array, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(audio_array))
        return np.hstack((mfccs, centroid, zcr))
    except:
        return None

X = []
y = []
SAMPLES_PER_DATASET = 2000

def consume_dataset(dataset, label, name, max_samples):
    if dataset is None: return
    print(f"\n⏬ Extracting Features from: {name}")
    count = 0
    try:
        for item in tqdm(dataset, total=max_samples):
            if count >= max_samples: break
            if 'audio' in item and 'array' in item['audio']:
                features = extract_features_from_array(item['audio']['array'], item['audio']['sampling_rate'])
                if features is not None:
                    X.append(features)
                    y.append(label)
                    count += 1
    except Exception as e:
        print(f"⚠️ Error extracting {name}: {e}")

consume_dataset(auth_1, 1, "Google Fleurs (Hindi)", SAMPLES_PER_DATASET)
consume_dataset(auth_2, 1, "IndicTTS-Hindi", SAMPLES_PER_DATASET)
consume_dataset(auth_3, 1, "UD-NLP Hindi Telephone", SAMPLES_PER_DATASET)
consume_dataset(fake_2, 0, "Hindi TTS Wikipedia Clones", SAMPLES_PER_DATASET)

X_final = np.array(X)
y_final = np.array(y)

if len(np.unique(y_final)) < 2:
    print("\n\n❌ FATAL ERROR: Missing Authentic or Deepfake data!")
else:
    print(f"\n✅ Successfully extracted features for {len(X_final)} total audio files!")

In [ ]:
# ==========================================
# 5. TRAIN XGBOOST DEEPFAKE MODEL
# ==========================================
if len(np.unique(y_final)) >= 2:
    print("\n🚀 Training XGBoost Classifier...")
    clf = xgb.XGBClassifier(n_estimators=300, max_depth=10, learning_rate=0.05, n_jobs=-1, random_state=42)
    clf.fit(X_final, y_final)
    
    print("\n✅ Training Complete! Accuracy Metrics:")
    predictions = clf.predict(X_final)
    print(classification_report(y_final, predictions, target_names=["Deepfake", "Authentic"]))
    
    with open("voiceguard_ultimate_model.pkl", "wb") as f:
        pickle.dump(clf, f)
        
    print("\n💾 SUCCESS! Ultimate Model saved as 'voiceguard_ultimate_model.pkl'")